# ThemeCloner V2 -- Walk-Forward RP-PCA

**Forked from ThemeCloner V1** (see `ThemeCloner/` repo for the submitted paper and its full single-snapshot diagnostic suite -- that pipeline is untouched).

This notebook implements the two changes scoped as ThemeCloner 2.0:
1. **Residualization keeps each stock's alpha** (`keep_alpha=True`) instead of forcing residuals to be mean-zero, so RP-PCA's premium-reward penalty has real cross-sectional mean-return information to exploit -- this is the actual mathematical difference between RP-PCA and plain PCA, which V1 confirmed collapses to PCA under mean-zero residuals (V1 paper, Section 3.3).
2. **Walk-forward, point-in-time estimation** (`src/rppca_walkforward.py`, `src/backtest_v2.py`): RP-PCA is refit at every rebalance date using only trailing data, theme fingerprints and candidate scores are built the same way, and baskets are held for a genuine forward period with realized (not in-sample) returns recorded. This closes the temporal look-ahead gap flagged in the V1 write-up: V1's "out-of-sample" validation was cross-sectional only (target-universe stocks held out) but the factor model itself was fit once on the full sample.

**Reused unchanged from V1:** `src/data_pull.py`, `src/rppca.py` (the core RP-PCA solver), `src/theme_dna.py` (`label_theme_factors`), `src/scoring.py` (`rank_candidates`), `src/theme_diagnostics.py`, `src/momentum_test.py`. Only `src/residualize.py` gained a new `keep_alpha` parameter (default `False`, so it is fully backward compatible with V1). Two new files, `src/rppca_walkforward.py` and `src/backtest_v2.py`, hold the walk-forward loop itself; `src/projection.py` gained new V2 scoring/null-test functions alongside its original V1 ones.

In [1]:
# -------------------- 0. Setup --------------------

import sys, os

# make sure we run from the ThemeCloner root regardless of where Jupyter launched
ROOT = os.path.abspath(os.path.join(os.getcwd()))
if not os.path.exists(os.path.join(ROOT, 'config', 'etfs.csv')):
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(ROOT)
sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

print(f'root: {ROOT}')
print(f'config exists: {os.path.exists("config/etfs.csv")}')
print('setup done')

root: C:\Users\aamin\ThemeCloner2
config exists: True
setup done


## Step 1 — Pull Data

In [2]:
# -------------------- 1a. Load ETF config --------------------

from src.data_pull import (load_etf_config, pull_etf_returns,
                            pull_covariance_universe, pull_target_universe)

# parameters -- tune these for the run
START_DATE  = '2018-01-01'
K           = 15     # factors for covariance universe RP-PCA (more themes need more K)
GAMMA       = 10.0   # Lettau-Pelger default
TOP_N       = 30     # candidates per theme
MIN_SCORE   = 0.40   # minimum cosine similarity to be a candidate
TOP_FACTORS = 3      # how many factors per ETF to use for theme fingerprint
RESIDUALIZE_MOMENTUM = False  # False = strip FF5 only (keep momentum)
INCLUDE_COMMODITY    = False  # EXPERIMENT 1: True = also strip commodity beta
                              # (DBC/DBE/DBB/GLD/DBA) to test if it rescues
                              # Clean Energy / Agribusiness from commodity contamination

etf_config = load_etf_config('config/etfs.csv')
etf_config


# split discovery themes from calibration controls (sector/subsector)
from src.data_pull import split_themes_controls
themes_config, controls_config = split_themes_controls(etf_config)
print(f'\n{len(themes_config["theme"].unique())} discovery themes, '
      f'{len(controls_config["theme"].unique())} calibration controls')

loaded 44 ETFs: 31 theme-ETFs, 13 control-ETFs (sector/subsector calibration anchors)
  AI Infrastructure: IGPT (IGPT), QTUM (QTUM), SOXX (SOXX)
  Agribusiness: MOO (MOO), VEGI (VEGI), PBJ (PBJ)
  Clean Energy: ICLN (ICLN), QCLN (QCLN), PBW (PBW)
  Critical Minerals: URA (URA), LIT (LIT), COPX (COPX)
  Cybersecurity: CIBR (CIBR), HACK (HACK), BUG (BUG)
  Defense: ITA (ITA), PPA (PPA), XAR (XAR)
  Factor1: IUSG (IUSG)
  Factor2: VLUE (VLUE)
  Factor3: QUAL (QUAL)
  Factor4: SPGP (SPGP)
  Factor5: MTUM (MTUM)
  Factor6: VUG (VUG)
  Factor7: VTV (VTV)
  Fintech: FINX (FINX), ARKF (ARKF), IPAY (IPAY)
  Robotics & AI: BOTZ (BOTZ), ROBO (ROBO), IRBO (IRBO)
  SECTOR: US Energy: XLE (XLE)
  SECTOR: US Financials: XLF (XLF)
  SECTOR: US Technology: XLK (XLK)
  SUBSECTOR: US Banks: KRE (KRE)
  SUBSECTOR: US Biotech: XBI (XBI)
  SUBSECTOR: US Software: IGV (IGV)
  Timber & Forestry: WOOD (WOOD), CUT (CUT)
  US Infrastructure: PAVE (PAVE), IFRA (IFRA)
  Water: PHO (PHO), FIW (FIW), CGW (CGW)

11 d

In [3]:
# -------------------- 1b. Pull ETF returns --------------------
# these serve two roles: theme definition (labeling factors) and OOS validation

etf_returns = pull_etf_returns(etf_config, start=START_DATE)
etf_returns.tail(3)


pulling ETF panel: 44 tickers in 1 batch(es)
  batch 1/1 (44 tickers)...
  dropped 1 tickers below 85% coverage
  ETF panel: 444 weeks x 43 stocks
  date range: 2018-01-12 to 2026-07-10
  ETFs kept: ['ARKF', 'BOTZ', 'CGW', 'CIBR', 'COPX', 'CUT', 'FINX', 'FIW', 'HACK', 'ICLN', 'IFRA', 'IGPT', 'IGV', 'IPAY', 'IRBO', 'ITA', 'IUSG', 'KRE', 'LIT', 'MOO', 'MTUM', 'PAVE', 'PBJ', 'PBW', 'PHO', 'PPA', 'QCLN', 'QTUM', 'QUAL', 'ROBO', 'SOXX', 'SPGP', 'URA', 'VEGI', 'VLUE', 'VTV', 'VUG', 'WOOD', 'XAR', 'XBI', 'XLE', 'XLF', 'XLK']


,ARKF,BOTZ,CGW,CIBR,COPX,CUT,FINX,FIW,HACK,ICLN,...,VEGI,VLUE,VTV,VUG,WOOD,XAR,XBI,XLE,XLF,XLK
Date,,,,,,,,,,,,,,,,,,,,,
2026-06-26,-0.020952,-0.063479,0.032378,0.010512,-0.115184,0.008190,-0.002807,0.033043,0.027230,-0.078859,...,0.014283,0.000850,0.013611,-0.048625,-0.007160,-0.036320,0.100083,0.008487,0.003497,-0.054278
2026-07-03,0.044163,0.038107,0.003196,0.060349,0.009543,0.002125,0.028280,-0.003937,0.080970,0.008680,...,0.010693,-0.036233,0.003565,0.032571,-0.000899,0.047975,0.032171,-0.011582,0.037554,-0.002875
2026-07-10,0.018859,-0.018490,0.001518,0.015757,-0.007464,-0.022820,-0.000391,-0.011720,0.020358,-0.029930,...,-0.003968,0.011464,-0.000434,0.013707,0.017526,-0.062251,-0.024414,0.030715,0.000719,0.018380


In [4]:
# -------------------- 1c. Pull covariance universe (ACWI proxy) --------------------
# this is what RP-PCA runs on -- needs to be broad enough to capture all themes
# SP500 + SP400 + SP600 + STOXX600 + Nikkei225
# first run: ~10-15 min to pull. subsequent runs use cache.

cov_returns = pull_covariance_universe(start=START_DATE, use_cache=True, us_only=True)
print(f'covariance universe: {cov_returns.shape[1]} stocks x {cov_returns.shape[0]} weeks')

loaded covariance universe from cache (us_only): 1003 tickers across ['us_sp500', 'us_mid_small']

pulling covariance universe: 1003 unique tickers

pulling covariance universe: 1003 tickers in 3 batch(es)
  batch 1/3 (400 tickers)...
  batch 2/3 (400 tickers)...
  batch 3/3 (203 tickers)...
  dropped 249 tickers below 85% coverage
  covariance universe: 444 weeks x 754 stocks
  date range: 2018-01-12 to 2026-07-10
covariance universe: 754 stocks x 444 weeks


In [5]:
# -------------------- 1d. Pull target universe (Russell 2000 proxy) --------------------
# small caps we score for undiscovered thematic exposure
# first run: ~20-30 min. subsequent runs use cache.

from src.data_pull import _cached_returns, pull_target_universe

#tgt_returns = _cached_returns('target_universe_returns', pull_target_universe, refresh=True, start=START_DATE, use_cache=False)
tgt_returns = _cached_returns('target_universe_returns', pull_target_universe, refresh=False, start=START_DATE)

print(f'target universe: {tgt_returns.shape[1]} stocks x {tgt_returns.shape[0]} weeks')

  loaded target_universe_returns from cache: 444 weeks x 811 cols
target universe: 811 stocks x 444 weeks


## Step 1.5 (V2) -- Residualize, keeping alpha for the covariance universe

In [6]:
# -------------------- 1.5 (V2). Residualize -- keep_alpha=True for the covariance universe --------------------
# THIS IS V2 CHANGE #1. Compare to V1's equivalent cell: the only difference
# is keep_alpha=True on cov_returns. ETFs and target universe stay V1-style
# (keep_alpha=False / default) -- see src/projection.py's V2 docstring for
# why that's fine: the V2 scoring metric demeans both series anyway, so it
# doesn't need alpha preserved on the ETF/target side, only on the
# covariance universe side where RP-PCA's mu term actually lives.

from src.residualize import get_combined_factors, residualize_returns, standardize_residuals

ff_factors = get_combined_factors(start=START_DATE, include_commodity=INCLUDE_COMMODITY)

print("\n" + "="*70)
print("RESIDUALIZING COVARIANCE UNIVERSE (keep_alpha=True -- V2 change #1)")
print("="*70)
cov_residuals_keepalpha = residualize_returns(cov_returns, ff_factors, keep_alpha=True)
cov_residuals_keepalpha = standardize_residuals(cov_residuals_keepalpha)

print("\n" + "="*70)
print("RESIDUALIZING ETF RETURNS (V1-style, keep_alpha=False)")
print("="*70)
etf_residuals = residualize_returns(etf_returns, ff_factors, keep_alpha=False)

print("\n" + "="*70)
print("RESIDUALIZING TARGET UNIVERSE (V1-style, keep_alpha=False)")
print("="*70)
tgt_residuals = residualize_returns(tgt_returns, ff_factors, keep_alpha=False)



pulling FF5 + momentum from Kenneth French library (weekly)...
  momentum fetch failed via pandas_datareader (Unknown datetime string format, unable to parse: Missing data are indicated by -99.99 or -999., at position 0)
  falling back to direct download from Ken French's data library...
  first 20 raw lines (for diagnosis):
    'This file was created by using the 202605 CRSP database.  It,,'
    'contains a momentum factor, constructed from six value-weight portfolios formed,'
    'using independent sorts on size and prior return of NYSE, AMEX, and NASDAQ stocks.'
    'MOM is the average of the returns on two (big and small) high prior return portfolios,,'
    'minus the average of the returns on two low prior return portfolios.  The portfolios,,'
    'are constructed daily.  Big means a firm is above the median market cap on the NYSE,,'
    'at the end of the previous day; small firms are below the median NYSE market cap.,,'
    'Prior return is measured from day - 250 to - 21.  Fir

## Step 2 (V2) -- Build the Rebalance Schedule

Quarterly rebalances, each requiring at least 2 years of trailing history before it's included. This directly answers the "can I keep rotating in and out of these positions" question from the V1 conversation -- instead of asserting stability from one fit, we now observe it, or don't, across real rebalance dates.

In [10]:
# -------------------- 2 (V2). Rebalance schedule --------------------
from src.rppca_walkforward import make_rebalance_dates

REBALANCE_FREQ = 'Q'          # quarterly
MIN_WINDOW_WEEKS = 104          # 2 years minimum trailing history
WALKFORWARD_WINDOW = 'expanding'  # 'expanding' or 'rolling'
ROLLING_WINDOW_WEEKS = 208      # only used if WALKFORWARD_WINDOW == 'rolling'

rebalance_dates = make_rebalance_dates(cov_residuals_keepalpha.index,
                                        freq=REBALANCE_FREQ,
                                        min_window_weeks=MIN_WINDOW_WEEKS)
print(f"{len(rebalance_dates)} rebalance dates:")
for d in rebalance_dates:
    print(f"  {d.date()}")


25 rebalance dates:
  2020-03-27
  2020-06-26
  2020-09-25
  2020-12-25
  2021-03-26
  2021-06-25
  2021-09-24
  2021-12-31
  2022-03-25
  2022-06-24
  2022-09-30
  2022-12-30
  2023-03-31
  2023-06-30
  2023-09-29
  2023-12-29
  2024-03-29
  2024-06-28
  2024-09-27
  2024-12-27
  2025-03-28
  2025-06-27
  2025-09-26
  2025-12-26
  2026-03-27


## Step 3 (V2) -- Factor Identity Drift

Before running the full backtest: how stable is each factor's economic identity from one rebalance to the next? This is computed directly from walk-forward RP-PCA fits, not asserted from a single snapshot.

In [11]:
# -------------------- 3 (V2). Walk-forward RP-PCA fits + factor drift --------------------
from src.rppca_walkforward import rolling_rppca_fit, factor_drift_report

K = 15
GAMMA = 10.0

walkforward_fits = rolling_rppca_fit(
    cov_residuals_keepalpha, rebalance_dates,
    K=K, gamma=GAMMA, window=WALKFORWARD_WINDOW,
    rolling_window_weeks=ROLLING_WINDOW_WEEKS,
)

drift = factor_drift_report(walkforward_fits)



[1/25] fitting RP-PCA as of 2020-03-27 (expanding, 116 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.676
  top eigenvalues:   [83.9 49.6 29.4 21.2 19.3 18.4 15.3 13.9 12.2 11.7 11.3 10.7  9.8  9.3
  9.1]

[2/25] fitting RP-PCA as of 2020-06-26 (expanding, 129 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.632
  top eigenvalues:   [79.4 42.9 29.6 24.8 22.6 18.5 17.5 15.4 14.2 13.  12.3 11.  10.7 10.5
 10.1]

[3/25] fitting RP-PCA as of 2020-09-25 (expanding, 142 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.847
  top eigenvalues:   [74.1 42.4 28.2 24.7 21.6 18.1 16.8 15.1 13.9 12.8 11.5 11.2 10.4 10.
  9.9]

[4/25] fitting RP-PCA as of 2020-12-25 (expanding, 155 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.279
  top eigenvalues:   [68.8 38.2 27.3 24.5 21.3 18.  16.1 14.6 13.5 12.3 11.  10.7 10.2  9.7
  9.4]

[5/25] fitting RP-PCA as of 2021-03-26 (expanding, 168 weeks x 7

## Step 4 (V2) -- Full Walk-Forward Backtest

This is the centerpiece. At every rebalance date: fit RP-PCA on trailing data only, build fingerprints on trailing ETF data only, score candidates on trailing target-universe data only, then hold the resulting basket for one forward period and record its REALIZED return -- data none of the prior steps ever saw.

In [ ]:
# -------------------- 4 (V2). Run the walk-forward backtest --------------------
from src.backtest_v2 import run_walkforward_backtest, summarize_backtest

TOP_N = 30
MIN_SCORE_V2 = -1.0   # V2 scores are correlations, can be legitimately negative;
                       # start permissive and tighten once score distributions are inspected
TOP_FACTORS = 3

wf_backtest = run_walkforward_backtest(
    cov_returns_resid_keepalpha=cov_residuals_keepalpha,
    etf_returns_resid=etf_residuals,
    target_returns_resid=tgt_residuals,
    target_returns_raw=tgt_returns,   # REAL returns for computing realized forward P&L
    etf_config=themes_config,
    rebalance_dates=rebalance_dates,
    K=K, gamma=GAMMA, window=WALKFORWARD_WINDOW,
    rolling_window_weeks=ROLLING_WINDOW_WEEKS,
    top_n=TOP_N, min_score=MIN_SCORE_V2, top_factors=TOP_FACTORS,
)

backtest_summary = summarize_backtest(wf_backtest['forward_returns'])



rebalance 1/25: 2020-03-27

[1/1] fitting RP-PCA as of 2020-03-27 (expanding, 116 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.676
  top eigenvalues:   [83.9 49.6 29.4 21.2 19.3 18.4 15.3 13.9 12.2 11.7 11.3 10.7  9.8  9.3
  9.1]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.692  f8:-0.002, f6:-0.002, f3:+0.002
  BOTZ                   0.519  f6:+0.001, f5:-0.001, f3:+0.001
  CGW                    0.726  f5:-0.001, f6:+0.001, f1:-0.001
  CIBR                   0.535  f4:-0.002, f9:+0.001, f13:+0.001
  COPX                   0.520  f2:-0.002, f1:+0.002, f8:+0.002
  CUT                    0.320  f12:-0.001, f6:+0.001, f8:+0.001
  FINX                   0.557  f3:+0.001, f5:+0.001, f1:-0.001
  FIW                    0.708  f6:+0.001, f8:+0.001, f1:-0.001
  HACK                   0.454  f4:-0.002, f9:+0.001, f13:+0.001
  ICLN 


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.537, 0.671]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 3/25: 2020-09-25

[1/1] fitting RP-PCA as of 2020-09-25 (expanding, 142 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.847
  top eigenvalues:   [74.1 42.4 28.2 24.7 21.6 18.1 16.8 15.1 13.9 12.8 11.5 11.2 10.4 10.
  9.9]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.551  f3:+0.003, f10:+0.002, f14:+0.001
  BOTZ                   0.489  f6:+0.002, f3:+0.001, f4:+0.001
  CGW                    0.655  f6:+0.001, f4:+0.001, f1:-0.0


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.539, 0.562]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 5/25: 2021-03-26

[1/1] fitting RP-PCA as of 2021-03-26 (expanding, 168 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.253
  top eigenvalues:   [66.6 35.5 26.3 24.  20.9 17.4 15.8 14.4 13.1 12.  10.8 10.7 10.   9.8
  9.5]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.571  f3:+0.003, f13:+0.002, f11:+0.001
  BOTZ                   0.455  f4:+0.001, f6:+0.001, f9:+0.001
  CGW                    0.610  f4:+0.001, f1:-0.001, f3:-0.


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.598, 0.493]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 7/25: 2021-09-24

[1/1] fitting RP-PCA as of 2021-09-24 (expanding, 194 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.277
  top eigenvalues:   [60.5 34.5 24.9 22.3 19.  16.  14.5 13.2 11.7 10.9  9.8  9.4  9.1  8.8
  8.5]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.529  f3:+0.003, f5:-0.001, f10:-0.001
  BOTZ                   0.398  f4:+0.002, f6:+0.001, f10:+0.001
  CGW                    0.584  f3:-0.001, f4:+0.001, f1:-0.


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.456, 0.443]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 9/25: 2022-03-25

[1/1] fitting RP-PCA as of 2022-03-25 (expanding, 220 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      5.867
  top eigenvalues:   [57.1 36.3 24.5 22.2 17.6 15.9 13.4 13.1 11.3 10.4  9.8  9.2  8.7  8.5
  8. ]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.484  f4:-0.002, f10:-0.002, f3:-0.002
  BOTZ                   0.350  f4:-0.001, f6:-0.001, f2:-0.001
  CGW                    0.611  f3:+0.001, f8:-0.001, f1:+0.0


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.353, 0.464]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 11/25: 2022-09-30

[1/1] fitting RP-PCA as of 2022-09-30 (expanding, 247 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      5.705
  top eigenvalues:   [55.5 35.9 24.  21.7 17.9 16.5 13.4 12.5 11.1 10.   9.4  9.2  8.6  8.4
  8.2]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.441  f10:+0.002, f6:-0.002, f2:-0.002
  BOTZ                   0.339  f2:-0.001, f4:-0.001, f5:+0.001
  CGW                    0.623  f1:+0.001, f3:+0.001, f5:+0.


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.388, 0.416]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 13/25: 2023-03-31

[1/1] fitting RP-PCA as of 2023-03-31 (expanding, 273 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      5.255
  top eigenvalues:   [53.4 32.  24.9 21.1 17.8 16.3 12.8 12.2 11.   9.8  9.2  9.   8.3  8.
  7.8]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.385  f3:-0.002, f11:-0.001, f4:-0.001
  BOTZ                   0.381  f4:-0.001, f12:-0.001, f5:+0.001
  CGW                    0.603  f5:+0.001, f1:+0.001, f4:-0.


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.356, 0.445]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 15/25: 2023-09-29

[1/1] fitting RP-PCA as of 2023-09-29 (expanding, 299 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      4.996
  top eigenvalues:   [54.4 29.8 23.9 19.9 17.1 15.7 12.3 11.6 10.6  9.5  8.9  8.6  8.1  7.7
  7.7]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.367  f3:-0.002, f5:-0.002, f10:+0.001
  BOTZ                   0.373  f4:-0.001, f2:-0.001, f10:-0.001
  CGW                    0.617  f4:-0.001, f1:-0.001, f13:+


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.343, 0.391]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 17/25: 2024-03-29

[1/1] fitting RP-PCA as of 2024-03-29 (expanding, 325 weeks x 754 stocks)
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      4.900
  top eigenvalues:   [52.3 27.7 22.7 21.1 17.2 15.6 11.7 11.4  9.9  9.   8.7  8.2  7.9  7.4
  7.3]

labeling theme factors using 43 ETFs:
  ETF                       R²  top factor weights
  ------------------------------------------------------------
  ARKF                   0.365  f4:-0.002, f5:-0.001, f13:+0.001
  BOTZ                   0.351  f4:-0.001, f2:+0.001, f6:-0.001
  CGW                    0.605  f13:+0.001, f1:-0.001, f5:+0


V2 scores (synthetic-factor-return correlation): 811 stocks x 11 themes
  range: [-0.376, 0.464]

ranked candidates summary:
theme
AI Infrastructure    30
Agribusiness         30
Clean Energy         30
Critical Minerals    30
Cybersecurity        30
Defense              30
Fintech              30
Robotics & AI        30
Timber & Forestry    30
US Infrastructure    30
Water                30

rebalance 19/25: 2024-09-27

[1/1] fitting RP-PCA as of 2024-09-27 (expanding, 351 weeks x 754 stocks)


## Step 5 (V2) -- Equity Curves

In [ ]:
# -------------------- 5 (V2). Equity curves --------------------
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 6))
wf_backtest['equity_curves'].plot(ax=ax)
ax.set_title('Walk-forward equity curves (genuinely out-of-sample, per theme)')
ax.set_ylabel('growth of $1')
ax.set_xlabel('rebalance date')
plt.tight_layout()
plt.savefig('outputs/fig_v2_walkforward_equity.png', dpi=150)
plt.show()


## Step 6 (V2) -- Candidate-Level Null Test (Most Recent Rebalance)

Directly answers: for the current period, is there actually genuine thematic exposure in the target universe for each theme, or would a random top-N look just as good? Uses the same point-in-time fit as the last backtest rebalance.

In [ ]:
# -------------------- 6 (V2). Candidate null test, most recent period --------------------
from src.projection import synthetic_theme_returns, score_universe_v2, candidate_null_test
from src.theme_dna import label_theme_factors

last_d = rebalance_dates[-1]
wf_last = walkforward_fits[last_d]
dna_last = label_theme_factors(wf_last, etf_residuals.loc[:last_d], themes_config,
                                top_factors=TOP_FACTORS)
synth_last = synthetic_theme_returns(wf_last, dna_last['theme_factors'])
scores_last = score_universe_v2(tgt_residuals.loc[:last_d], synth_last)

null_test = candidate_null_test(scores_last, top_n=TOP_N, n_null_draws=1000)


### Commit to Git

In [ ]:
%cd "C:\\Users\\aamin\\ThemeCloner2"

!git add .
!git commit -m "walk-forward V2 run"
!git push origin master
